# Real-Estate Classifier - Training Walkthrough

Notebook paso a paso equivalente al flujo de `src/prepare_data.py` + `src/train.py`.

Objetivo: explorar y entrenar de forma interactiva sin perder trazabilidad en W&B.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Root:", ROOT)
print("Src added:", SRC.exists())

In [ ]:
import torch
import wandb

from real_estate_ml.config import load_config
from real_estate_ml.constants import CLASSES
from real_estate_ml.data.prepare_splits import prepare_splits
from real_estate_ml.data.dataset import get_dataloaders
from real_estate_ml.models.classifier import build_model
from real_estate_ml.training.engine import run_epoch

cfg = load_config(ROOT / "configs" / "base_config.yaml")
cfg

In [ ]:
# 1) Preparar splits de forma idempotente (sin duplicados)
prepare_splits(
    raw_training_dir=ROOT / cfg["data"]["raw_training_dir"],
    raw_validation_dir=ROOT / cfg["data"]["raw_validation_dir"],
    output_dir=ROOT / cfg["data"]["data_dir"],
    train_split=cfg["data"]["train_split"],
    val_split=cfg["data"]["val_split"],
    test_split=cfg["data"]["test_split"],
    seed=cfg["training"]["seed"],
)

In [ ]:
# 2) Dataloaders + modelo + optimización
requested_device = cfg["hardware"].get("device", "cpu")
device = torch.device("cuda" if requested_device == "cuda" and torch.cuda.is_available() else "cpu")
print("Device:", device)

dataloaders = get_dataloaders(
    data_dir=str(ROOT / cfg["data"]["data_dir"]),
    batch_size=cfg["data"]["batch_size"],
    image_size=cfg["data"]["image_size"],
    num_workers=cfg["data"]["num_workers"],
)

model = build_model(
    backbone=cfg["model"]["backbone"],
    num_classes=cfg["data"]["num_classes"],
    pretrained=cfg["model"]["pretrained"],
    dropout=cfg["model"]["dropout"],
    freeze_backbone=cfg["model"]["freeze_backbone"],
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg["training"]["learning_rate"]),
    weight_decay=float(cfg["training"]["weight_decay"]),
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["training"]["epochs"])

In [ ]:
# 3) Inicializar W&B (si quieres desactivarlo, usa mode='disabled')
run = wandb.init(
    project=cfg["project_name"],
    entity=cfg.get("entity"),
    config=cfg,
    job_type="train-notebook",
)
run.name

In [ ]:
# 4) Entrenamiento por épocas (visible paso a paso)
from pathlib import Path

best_macro_f1 = -1.0
patience = 0
save_dir = ROOT / cfg["training"]["save_dir"]
save_dir.mkdir(parents=True, exist_ok=True)
best_model_path = save_dir / "best_model.pth"

for epoch in range(cfg["training"]["epochs"]):
    print(f"\nEpoch {epoch + 1}/{cfg['training']['epochs']}")
    train_metrics = run_epoch(model, dataloaders["train"], criterion, optimizer, device, train=True)
    val_metrics = run_epoch(model, dataloaders["val"], criterion, optimizer, device, train=False)
    scheduler.step()

    wandb.log(
        {
            "epoch": epoch + 1,
            "train/loss": train_metrics.loss,
            "train/macro_f1": train_metrics.macro_f1,
            "val/loss": val_metrics.loss,
            "val/macro_f1": val_metrics.macro_f1,
            "lr": optimizer.param_groups[0]["lr"],
        }
    )

    print(
        f"train_loss={train_metrics.loss:.4f} train_f1={train_metrics.macro_f1:.4f} "
        f"val_loss={val_metrics.loss:.4f} val_f1={val_metrics.macro_f1:.4f}"
    )

    if val_metrics.macro_f1 > best_macro_f1:
        best_macro_f1 = val_metrics.macro_f1
        patience = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "backbone": cfg["model"]["backbone"],
                "num_classes": cfg["data"]["num_classes"],
                "classes": CLASSES,
            },
            best_model_path,
        )
        print(f"New best model saved -> {best_model_path}")
    else:
        patience += 1

    if patience >= cfg["training"]["early_stopping_patience"]:
        print("Early stopping triggered.")
        break

print("Best val macro_f1:", best_macro_f1)

In [ ]:
# 5) Evaluación en test + matriz de confusión
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics = run_epoch(model, dataloaders["test"], criterion, optimizer=None, device=device, train=False)

print("test_loss:", round(test_metrics.loss, 4))
print("test_macro_f1:", round(test_metrics.macro_f1, 4))

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(test_metrics.confusion_matrix, display_labels=CLASSES).plot(
    xticks_rotation=45,
    ax=ax,
    colorbar=False,
)
plt.tight_layout()

wandb.log(
    {
        "test/loss": test_metrics.loss,
        "test/macro_f1": test_metrics.macro_f1,
        "test/confusion_matrix": wandb.Image(fig),
    }
)
plt.show()

In [ ]:
# 6) Guardar artifact y cerrar run
artifact = wandb.Artifact("best-model", type="model")
artifact.add_file(str(best_model_path))
run.log_artifact(artifact)
run.finish()

print("Training notebook complete. Model:", best_model_path)

## Notes

- Si quieres iterar rápido, baja `cfg['training']['epochs']` antes de ejecutar la celda de entrenamiento.
- Puedes cambiar `cfg['model']['backbone']` en caliente para comparar modelos.
- Este notebook genera el mismo checkpoint que la versión script (`artifacts/best_model.pth`).